In [1]:
!pip install git+https://github.com/huggingface/transformers.git
!pip install accelerate

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-orfystrj
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-orfystrj
  Resolved https://github.com/huggingface/transformers.git to commit b84c0b31c674436b076bba818885226a0e0ddecc
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.0/502.0 kB 10.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 67.5 MB/s eta 0:00:00:00:01
  Created wheel for transformers: filename=transformers-5.0.0.dev0-py3-none-any.whl size=11390241 sha256=95e85ea3d2c1f8e3167c3698e096d31a732c19397c1107a86d0b588bc5885536
  Stored in directory: /tmp/pip-ephem-wheel-cache-03i4xue9/wheels/32/4b/78/f195c684dd3a9ed21f3b39fe8f85b48df7918581b6437be143
Successfully built transformers
  Attempting uninstall: huggingfac

In [4]:
# =============================================================================
# 1. IMPORTS & SETUP
# =============================================================================
import os
import re
import pandas as pd
import polars as pl
import torch
from tqdm import tqdm
from sklearn.metrics import classification_report, accuracy_score

# Use the pipeline function for a higher-level interface
from transformers import pipeline

# Access Hugging Face token from Kaggle Secrets
from kaggle_secrets import UserSecretsClient
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
except:
    print("HF_TOKEN secret not found.")
    os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN_HERE"


# =============================================================================
# 2. CONFIGURATION
# =============================================================================
class Config:
    # --- Model Configuration ---
    # Using the model name from the official code snippet
    MODEL_NAME = "Equall/Saul-Instruct-v1"

    # --- Data Paths ---
    POS_DATA_FILE = "/kaggle/input/lrec-dataset/sentencePair.txt"
    NEG_DATA_FILE = "/kaggle/input/lrec-dataset/sentencePair_neg.txt"
    OUTPUT_FILE = "/kaggle/working/zero_shot_results_instruct.csv"

    # --- Inference Configuration ---
    SAMPLE_SIZE = 500
    BATCH_SIZE = 4 # You can increase this with pipelines, e.g., to 8

    # --- Environment ---
    DEVICE = 0 # Pipelines manage device mapping, we can specify the primary GPU


# =============================================================================
# 3. DATA LOADING & PREPARATION (Unchanged from last version)
# =============================================================================
FILENAME_PATTERN = re.compile(r"SupremeCourt_(\d{4})_(\d+)\.txt")

def parse_lrec_line(line: str):
    parts = line.strip().split("\t")
    fname_indices = [i for i, p in enumerate(parts) if p.endswith(".txt")]
    if len(fname_indices) != 2: return None
    try:
        sentpair_id = int(parts[0])
        sent1 = " ".join(parts[fname_indices[0] + 2 : fname_indices[1]]).strip('"')
        sent2 = " ".join(parts[fname_indices[1] + 2 : len(parts) - 2]).strip('"')
        label = parts[-1]
    except (ValueError, IndexError): return None
    return {"id": sentpair_id, "sent1": sent1, "sent2": sent2, "label": label}

def load_lrec_file(filepath: str) -> pd.DataFrame:
    rows = []
    print(f"Reading data from {filepath}...")
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            if parsed_row := parse_lrec_line(line):
                rows.append(parsed_row)
    return pd.DataFrame(rows)

def load_and_split_data(config: Config):
    print("Loading and splitting data...")
    df_pos = load_lrec_file(config.POS_DATA_FILE)
    df_neg = load_lrec_file(config.NEG_DATA_FILE)
    if df_pos.empty or df_neg.empty:
        print("ERROR: Data files failed to load.")
        return None
    df_pos_sample = df_pos.sample(n=min(10000, len(df_pos)), random_state=42)
    df_neg_sample = df_neg.sample(n=min(10000, len(df_neg)), random_state=42)
    remaining_pos = df_pos.drop(df_pos_sample.index)
    remaining_neg = df_neg.drop(df_neg_sample.index)
    df_test = pd.concat([remaining_pos, remaining_neg]).reset_index(drop=True)
    df_test = df_test.dropna(subset=['id', 'sent1', 'sent2', 'label'])
    print(f"Created test set with {len(df_test)} samples.")
    if config.SAMPLE_SIZE > 0:
        print(f"Using a sample of {config.SAMPLE_SIZE} examples from the test set for evaluation.")
        df_test = df_test.sample(n=min(config.SAMPLE_SIZE, len(df_test)), random_state=42).reset_index(drop=True)
    return df_test

# =============================================================================
# 4. ZERO-SHOT INFERENCE PIPELINE (REWRITTEN)
# =============================================================================
class ZeroShotClassifierWithPipeline:
    def __init__(self, config: Config):
        self.config = config
        self.pipe = self._load_pipeline()
        if self.pipe.tokenizer.pad_token_id is None:
            self.pipe.tokenizer.pad_token_id = self.pipe.model.config.eos_token_id

    def _load_pipeline(self):
        print(f"Loading pipeline for model {self.config.MODEL_NAME}...")
        # This single command initializes the model, tokenizer, and puts it on the GPU
        return pipeline(
            "text-generation",
            model=self.config.MODEL_NAME,
            dtype=torch.bfloat16,
            device_map="auto",
            trust_remote_code=True # Good practice to keep this
        )

    def create_message(self, sent1: str, sent2: str) -> list:
        """Creates the message structure for the chat template."""
        content = f"""Analyze the relationship between the following two sentences. Classify the relationship as one of three categories: SUPPORT, ATTACK, or NO RELATION. Provide only the label as your answer.

Sentence 1: "{sent1}"
Sentence 2: "{sent2}"
"""
        return [{"role": "user", "content": content}]

    def parse_output(self, generated_text: str) -> str:
        """Extracts the label from the model's free-text response."""
        output = generated_text.strip().upper()
        if "SUPPORT" in output: return "SUPPORT"
        if "ATTACK" in output: return "ATTACK"
        if "NO RELATION" in output or "NO_REL" in output: return "NO_REL"
        return "UNKNOWN"

    def predict(self, df: pd.DataFrame):
        """Runs batch inference on the entire dataframe using the pipeline."""
        all_messages = [self.create_message(row.sent1, row.sent2) for _, row in df.iterrows()]
        
        # Format all messages using the model's specific chat template
        prompts = [
            self.pipe.tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
            for msg in all_messages
        ]
        
        all_predictions = []
        print(f"Starting inference on {len(prompts)} samples...")
        
        # The pipeline can process a list of prompts in a batch
        outputs = self.pipe(
            prompts,
            max_new_tokens=100,
            do_sample=False,
            batch_size=self.config.BATCH_SIZE,
            return_full_text=False # Only return the generated part
        )
        
        # Extract and parse the generated text from each output
        for output in outputs:
            generated_text = output[0]['generated_text']
            prediction = self.parse_output(generated_text)
            all_predictions.append(prediction)
            
        return all_predictions

# =============================================================================
# 5. MAIN EXECUTION BLOCK
# =============================================================================
if __name__ == '__main__':
    config = Config()
    df_test = load_and_split_data(config)

    if df_test is not None:
        classifier = ZeroShotClassifierWithPipeline(config)
        predictions = classifier.predict(df_test)

        # --- EVALUATION ---
        print("\n" + "="*50)
        print("      ZERO-SHOT CLASSIFICATION ON TEST SET")
        print("="*50)
        true_labels = df_test['label']
        accuracy = accuracy_score(true_labels, predictions)
        print(f"\nOverall Accuracy: {accuracy:.4f}\n")
        print("Classification Report (Precision, Recall, F1-Score):")
        report = classification_report(
            true_labels, predictions, labels=list(true_labels.unique()), zero_division=0
        )
        print(report)

        # --- SAVING THE OUTPUT ---
        output_df = pd.DataFrame({
            'pair_id': df_test['id'],
            'sentence_1': df_test['sent1'],
            'sentence_2': df_test['sent2'],
            'original_label': df_test['label'],
            'llm_generated_label': predictions
        })
        output_df.to_csv(config.OUTPUT_FILE, index=False)
        print(f"\n Results saved successfully to {config.OUTPUT_FILE}")

        # Display some examples
        print("\n--- Sample Predictions ---")
        print(output_df.head(10))

Loading and splitting data...
Reading data from /kaggle/input/lrec-dataset/sentencePair.txt...
Reading data from /kaggle/input/lrec-dataset/sentencePair_neg.txt...
Created test set with 20506 samples.
Using a sample of 500 examples from the test set for evaluation.
Loading pipeline for model Equall/Saul-Instruct-v1...


config.json:   0%|          | 0.00/653 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

model-00001-of-00006.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Starting inference on 500 samples...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o


      ZERO-SHOT CLASSIFICATION ON TEST SET

Overall Accuracy: 0.5400

Classification Report (Precision, Recall, F1-Score):
              precision    recall  f1-score   support

      NO_REL       0.74      0.66      0.70       247
      ATTACK       0.40      0.03      0.06       130
     SUPPORT       0.38      0.84      0.52       123

    accuracy                           0.54       500
   macro avg       0.51      0.51      0.43       500
weighted avg       0.56      0.54      0.49       500


 Results saved successfully to /kaggle/working/zero_shot_results_instruct.csv

--- Sample Predictions ---
   pair_id                                         sentence_1  \
0    15136  It was , however , held that in case the Junio...   
1     4842  However , the charges Nos. II , V and X were e...   
2    10249  For the price of goods is affected by many fac...   
3     9450  Mr. Andley also attempted to argue that the de...   
4    15504  Learned counsel for the petitioners emphatical...  

In [2]:
# =============================================================================
# 1. IMPORTS & SETUP
# =============================================================================
import os
import re
import pandas as pd
import polars as pl
import torch
from tqdm import tqdm
from sklearn.metrics import classification_report, accuracy_score

# Use the pipeline function for a higher-level interface
from transformers import pipeline

# Access Hugging Face token from Kaggle Secrets
from kaggle_secrets import UserSecretsClient
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
except:
    print("HF_TOKEN secret not found.")
    os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN_HERE"


# =============================================================================
# 2. CONFIGURATION
# =============================================================================
class Config:
    # --- Model Configuration ---
    MODEL_NAME = "Equall/Saul-Instruct-v1"

    # --- Data Paths ---
    POS_DATA_FILE = "/kaggle/input/lrec-dataset/sentencePair.txt"
    NEG_DATA_FILE = "/kaggle/input/lrec-dataset/sentencePair_neg.txt"
    OUTPUT_FILE = "/kaggle/working/zero_shot_results_instruct.csv"

    # --- Inference Configuration ---
    SAMPLE_SIZE = 500
    BATCH_SIZE = 4  # You can increase this with pipelines, e.g., to 8
    MAX_NEW_TOKENS = 32  # label-only, but keep a small safety margin

    # --- Environment ---
    DEVICE = 0  # Pipelines manage device mapping, we can specify the primary GPU

    # --- Few-shot Configuration ---
    N_SHOTS = 6                 # total exemplars to prepend
    SHOT_STRATEGY = "balanced"  # "balanced" or "random"
    RANDOM_SEED = 42
    MAX_INPUT_TOKENS = 2048     # truncate long prompts if needed


# =============================================================================
# 3. DATA LOADING & PREPARATION
# =============================================================================
FILENAME_PATTERN = re.compile(r"SupremeCourt_(\d{4})_(\d+)\.txt")

def parse_lrec_line(line: str):
    parts = line.strip().split("\t")
    fname_indices = [i for i, p in enumerate(parts) if p.endswith(".txt")]
    if len(fname_indices) != 2: return None
    try:
        sentpair_id = int(parts[0])
        sent1 = " ".join(parts[fname_indices[0] + 2 : fname_indices[1]]).strip('"')
        sent2 = " ".join(parts[fname_indices[1] + 2 : len(parts) - 2]).strip('"')
        label = parts[-1]
    except (ValueError, IndexError): return None
    return {"id": sentpair_id, "sent1": sent1, "sent2": sent2, "label": label}

def load_lrec_file(filepath: str) -> pd.DataFrame:
    rows = []
    print(f"Reading data from {filepath}...")
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            if parsed_row := parse_lrec_line(line):
                rows.append(parsed_row)
    return pd.DataFrame(rows)

def load_and_split_data(config: Config):
    """
    Returns:
      df_test: test dataframe
      df_shots: pool from which few-shot exemplars are drawn (disjoint from df_test)
    """
    print("Loading and splitting data...")
    df_pos = load_lrec_file(config.POS_DATA_FILE)
    df_neg = load_lrec_file(config.NEG_DATA_FILE)
    if df_pos.empty or df_neg.empty:
        print("ERROR: Data files failed to load.")
        return None, None

    # Few-shot pool (sample a chunk to keep memory & speed reasonable)
    df_pos_pool = df_pos.sample(n=min(10000, len(df_pos)), random_state=config.RANDOM_SEED)
    df_neg_pool = df_neg.sample(n=min(10000, len(df_neg)), random_state=config.RANDOM_SEED)
    df_shots = pd.concat([df_pos_pool, df_neg_pool]).reset_index(drop=True)

    # Test set excludes the shot pool to avoid leakage
    remaining_pos = df_pos.drop(df_pos_pool.index)
    remaining_neg = df_neg.drop(df_neg_pool.index)
    df_test = pd.concat([remaining_pos, remaining_neg]).reset_index(drop=True)
    df_test = df_test.dropna(subset=['id', 'sent1', 'sent2', 'label'])
    print(f"Created test set with {len(df_test)} samples.")

    if config.SAMPLE_SIZE > 0:
        print(f"Using a sample of {config.SAMPLE_SIZE} examples from the test set for evaluation.")
        df_test = df_test.sample(n=min(config.SAMPLE_SIZE, len(df_test)), random_state=config.RANDOM_SEED).reset_index(drop=True)

    return df_test, df_shots


# =============================================================================
# 4. FEW-SHOT HELPERS
# =============================================================================
def _norm_label(s: str) -> str:
    return s.strip().upper().replace(" ", "_")

def build_exemplars_pool(df_shots: pd.DataFrame, k: int, strategy: str, rng_seed: int):
    df = df_shots.copy()
    df["label_norm"] = df["label"].apply(_norm_label)

    def pick_balanced():
        df_bal = df.copy()
        df_bal.loc[df_bal["label_norm"] == "NO_RELATION", "label_norm"] = "NO_REL"
        per = max(1, k // 3)
        buckets = []
        for lab in ["SUPPORT", "ATTACK", "NO_REL"]:
            sub = df_bal[df_bal["label_norm"] == lab]
            if len(sub) > 0:
                buckets.append(sub.sample(n=min(per, len(sub)), random_state=rng_seed))
        out = pd.concat(buckets) if buckets else df.sample(n=min(k, len(df)), random_state=rng_seed)
        if len(out) < k and len(df) > len(out):
            remain = df.drop(out.index)
            extra = remain.sample(n=min(k - len(out), len(remain)), random_state=rng_seed)
            out = pd.concat([out, extra])
        return out.sample(frac=1.0, random_state=rng_seed).head(min(k, len(out)))

    def pick_random():
        return df.sample(n=min(k, len(df)), random_state=rng_seed)

    strat_fn = pick_balanced if strategy == "balanced" else pick_random

    def chooser():
        chosen = strat_fn()
        return [
            {
                "sent1": r["sent1"],
                "sent2": r["sent2"],
                "label": "NO_REL" if _norm_label(r["label"]) in ("NO_REL", "NO_RELATION") else _norm_label(r["label"]),
            }
            for _, r in chosen.iterrows()
        ]
    return chooser

def make_examples_block(exemplars):
    lines = ["Here are labeled examples:\n"]
    for i, ex in enumerate(exemplars, 1):
        lines.append(
            f"Example {i}:\n"
            f"Sentence 1: \"{ex['sent1']}\"\n"
            f"Sentence 2: \"{ex['sent2']}\"\n"
            f"Label: {ex['label']}\n"
        )
    lines.append(
        "Labels are one of: SUPPORT, ATTACK, NO_REL.\n"
        "Only output the label for the next pair."
    )
    return "\n".join(lines)


# =============================================================================
# 5. FEW-SHOT INFERENCE PIPELINE
# =============================================================================
class FewShotClassifierWithPipeline:
    def __init__(self, config: Config, df_shots: pd.DataFrame):
        self.config = config
        self.pipe = self._load_pipeline()
        if self.pipe.tokenizer.pad_token_id is None:
            self.pipe.tokenizer.pad_token_id = self.pipe.model.config.eos_token_id

        # Build ONE fixed exemplar set for the whole run (stable & shorter prompts overall)
        self.exemplars = build_exemplars_pool(
            df_shots=df_shots,
            k=config.N_SHOTS,
            strategy=config.SHOT_STRATEGY,
            rng_seed=config.RANDOM_SEED
        )()
        self.examples_block = make_examples_block(self.exemplars)

    def _load_pipeline(self):
        print(f"Loading pipeline for model {self.config.MODEL_NAME}...")
        return pipeline(
            "text-generation",
            model=self.config.MODEL_NAME,
            dtype=torch.bfloat16,     # keep your working dtype
            device_map="auto",
            trust_remote_code=True    # keep this as you had it
        )

    def create_message(self, sent1: str, sent2: str) -> list:
        """Few-shot prompt with in-context exemplars, then the query pair."""
        content = (
            "You are a classifier for argumentative relations between two sentences.\n"
            "Given two sentences, classify the relation as exactly one of: SUPPORT, ATTACK, or NO_REL.\n\n"
            f"{self.examples_block}\n\n"
            "Now classify the following pair. Respond with ONLY the label (no punctuation or explanation):\n\n"
            f"Sentence 1: \"{sent1}\"\n"
            f"Sentence 2: \"{sent2}\"\n"
        )
        return [{"role": "user", "content": content}]

    def parse_output(self, generated_text: str) -> str:
        """Extracts the label from the model's free-text response."""
        output = generated_text.strip().upper()
        if "SUPPORT" in output: return "SUPPORT"
        if "ATTACK" in output: return "ATTACK"
        if "NO RELATION" in output or "NO_REL" in output: return "NO_REL"
        return "NO_REL"  # gentle fallback improves stability

    def predict(self, df: pd.DataFrame):
        """Runs batch inference on the entire dataframe using the pipeline."""
        all_messages = [self.create_message(row.sent1, row.sent2) for _, row in df.iterrows()]

        # Format all messages using the model's chat template
        prompts = [
            self.pipe.tokenizer.apply_chat_template(
                msg, tokenize=False, add_generation_prompt=True, truncation=True, max_length=self.config.MAX_INPUT_TOKENS
            )
            for msg in all_messages
        ]

        all_predictions = []
        print(f"Starting few-shot inference on {len(prompts)} samples (shots={self.config.N_SHOTS})...")

        outputs = self.pipe(
            prompts,
            max_new_tokens=self.config.MAX_NEW_TOKENS,
            do_sample=False,
            batch_size=self.config.BATCH_SIZE,
            return_full_text=False  # Only return the generated part
        )

        for output in outputs:
            generated_text = output[0]['generated_text'] if isinstance(output, list) else str(output)
            prediction = self.parse_output(generated_text)
            all_predictions.append(prediction)

        return all_predictions


# =============================================================================
# 6. MAIN EXECUTION BLOCK
# =============================================================================
if __name__ == '__main__':
    config = Config()
    df_test, df_shots = load_and_split_data(config)

    if df_test is not None and df_shots is not None and not df_shots.empty:
        classifier = FewShotClassifierWithPipeline(config, df_shots=df_shots)
        predictions = classifier.predict(df_test)

        # --- EVALUATION ---
        print("\n" + "="*50)
        print(f"      FEW-SHOT CLASSIFICATION ON TEST SET (k={config.N_SHOTS})")
        print("="*50)
        # normalize true labels to match our three-class scheme
        true_labels = df_test['label'].str.upper().str.replace(" ", "_").replace({"NO_RELATION": "NO_REL"})
        accuracy = accuracy_score(true_labels, predictions)
        print(f"\nOverall Accuracy: {accuracy:.4f}\n")
        print("Classification Report (Precision, Recall, F1-Score):")
        report = classification_report(
            true_labels, predictions, labels=sorted(true_labels.unique()), zero_division=0
        )
        print(report)

        # --- SAVING THE OUTPUT ---
        output_df = pd.DataFrame({
            'pair_id': df_test['id'],
            'sentence_1': df_test['sent1'],
            'sentence_2': df_test['sent2'],
            'original_label': df_test['label'],
            'llm_generated_label': predictions
        })
        output_df.to_csv(config.OUTPUT_FILE, index=False)
        print(f"\n Results saved successfully to {config.OUTPUT_FILE}")

        # Display some examples
        print("\n--- Sample Predictions ---")
        print(output_df.head(10))
    else:
        print("Unable to run: test set or shot pool is empty.")


Loading and splitting data...
Reading data from /kaggle/input/lrec-dataset/sentencePair.txt...
Reading data from /kaggle/input/lrec-dataset/sentencePair_neg.txt...
Created test set with 20506 samples.
Using a sample of 500 examples from the test set for evaluation.
Loading pipeline for model Equall/Saul-Instruct-v1...


config.json:   0%|          | 0.00/653 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

model-00002-of-00006.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Starting few-shot inference on 500 samples (shots=6)...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o


      FEW-SHOT CLASSIFICATION ON TEST SET (k=6)

Overall Accuracy: 0.5320

Classification Report (Precision, Recall, F1-Score):
              precision    recall  f1-score   support

      ATTACK       0.20      0.01      0.01       130
      NO_REL       0.53      0.93      0.68       247
     SUPPORT       0.55      0.29      0.38       123

    accuracy                           0.53       500
   macro avg       0.43      0.41      0.36       500
weighted avg       0.45      0.53      0.43       500


 Results saved successfully to /kaggle/working/zero_shot_results_instruct.csv

--- Sample Predictions ---
   pair_id                                         sentence_1  \
0    15136  It was , however , held that in case the Junio...   
1     4842  However , the charges Nos. II , V and X were e...   
2    10249  For the price of goods is affected by many fac...   
3     9450  Mr. Andley also attempted to argue that the de...   
4    15504  Learned counsel for the petitioners emphatical